# Complete End-to-End RAG Evaluation Pipeline

This notebook ties **everything together** into a production-ready evaluation pipeline. It demonstrates how to:

1. **Define a configurable RAG pipeline** (Qdrant + OpenAI)
2. **Create a comprehensive evaluation dataset** (manual + synthetic)
3. **Define a full metrics suite** (DeepEval + RAGAS + custom)
4. **Run full evaluation** across both frameworks
5. **Analyze results** with statistics, correlations, and visualizations
6. **Run hyperparameter experiments** (chunk size, top-K, temperature)
7. **Set up regression testing** with baselines
8. **Generate a summary report**

This is a self-contained, production-quality notebook.

**Prerequisites**: All prior notebooks (01-09) completed, OpenAI API key in `.env`

In [ ]:
import os
import json
import time
import hashlib
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print(f"Pipeline initialized at {datetime.now().isoformat()}")
print(f"Working directory: {os.getcwd()}")

-
## Step 1: Define the Configurable RAG Pipeline

We build the RAG pipeline as a **configurable class** so we can easily test different parameter combinations.

In [ ]:
from dataclasses import dataclass


@dataclass
class RAGConfig:
    """Configuration for the RAG pipeline."""
    chunk_size: int = 500          # Characters per chunk
    chunk_overlap: int = 50        # Overlap between chunks
    top_k: int = 3                 # Number of chunks to retrieve
    embedding_model: str = "text-embedding-3-small"
    generation_model: str = "gpt-4o-mini"
    temperature: float = 0.0
    system_prompt: str = (
        "You are a helpful customer support assistant for Acme Corp. "
        "Answer questions based on the provided context. If the answer is not in the "
        "context, say so. Be concise and accurate."
    )
    
    @property
    def name(self) -> str:
        return f"chunk{self.chunk_size}_top{self.top_k}_temp{self.temperature}"


class RAGPipeline:
    """Configurable RAG pipeline with Qdrant + OpenAI."""
    
    EMBEDDING_DIM = 1536
    
    def __init__(self, config: RAGConfig, documents: list[dict]):
        self.config = config
        self.openai_client = OpenAI()
        self.qdrant_client = QdrantClient(":memory:")
        self.collection_name = f"rag_{config.name}"
        self.documents = documents
        self._build_index()
    
    def _get_embeddings(self, texts: list[str]) -> list[list[float]]:
        """Get embeddings for a list of texts."""
        response = self.openai_client.embeddings.create(
            input=texts, model=self.config.embedding_model
        )
        return [item.embedding for item in response.data]
    
    def _chunk_text(self, text: str) -> list[str]:
        """Split text into chunks with overlap."""
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.config.chunk_size
            chunks.append(text[start:end])
            start = end - self.config.chunk_overlap
        return chunks
    
    def _build_index(self):
        """Chunk documents, embed, and store in Qdrant."""
        all_chunks = []
        for doc in self.documents:
            full_text = f"{doc['title']}\n\n{doc['content']}"
            if len(full_text) > self.config.chunk_size:
                doc_chunks = self._chunk_text(full_text)
            else:
                doc_chunks = [full_text]
            for chunk in doc_chunks:
                all_chunks.append({"text": chunk, "title": doc["title"]})
        
        # Embed
        texts = [c["text"] for c in all_chunks]
        embeddings = self._get_embeddings(texts)
        
        # Create collection
        self.qdrant_client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=self.EMBEDDING_DIM, distance=Distance.COSINE),
        )
        
        # Upsert
        points = [
            PointStruct(id=i+1, vector=emb, payload=chunk)
            for i, (chunk, emb) in enumerate(zip(all_chunks, embeddings))
        ]
        self.qdrant_client.upsert(collection_name=self.collection_name, points=points)
        self.num_chunks = len(points)
    
    def retrieve(self, query: str) -> list[str]:
        """Retrieve top-k relevant chunks."""
        query_embedding = self._get_embeddings([query])[0]
        results = self.qdrant_client.query_points(
            collection_name=self.collection_name,
            query=query_embedding,
            limit=self.config.top_k,
        )
        return [hit.payload["text"] for hit in results.points]
    
    def generate(self, query: str, contexts: list[str]) -> str:
        """Generate answer from query and contexts."""
        context_str = "\n\n---\n\n".join(contexts)
        messages = [
            {"role": "system", "content": self.config.system_prompt},
            {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
        ]
        response = self.openai_client.chat.completions.create(
            model=self.config.generation_model,
            messages=messages,
            temperature=self.config.temperature,
        )
        return response.choices[0].message.content
    
    def run(self, query: str) -> dict:
        """Full RAG pipeline: retrieve then generate."""
        start = time.time()
        contexts = self.retrieve(query)
        answer = self.generate(query, contexts)
        duration_ms = (time.time() - start) * 1000
        return {
            "query": query,
            "answer": answer,
            "contexts": contexts,
            "latency_ms": duration_ms,
        }


print("RAGPipeline class defined.")

In [ ]:
# Load knowledge base from file (created in Notebook 02)
kb_path = os.path.join(os.getcwd(), "data", "knowledge_base.json")

if os.path.exists(kb_path):
    with open(kb_path) as f:
        DOCUMENTS = json.load(f)
    print(f"Loaded {len(DOCUMENTS)} documents from {kb_path}")
    # Normalize key names: the file uses 'text' but the pipeline expects 'content'
    for doc in DOCUMENTS:
        if "text" in doc and "content" not in doc:
            doc["content"] = doc.pop("text")
    print(f"Document keys: {list(DOCUMENTS[0].keys())}")
else:
    # Fallback: inline documents
    print("knowledge_base.json not found, using inline documents.")
    DOCUMENTS = [
        {"id": 1, "title": "Return Policy Overview", "content":
         "Acme Corp offers a 30-day return policy for most items. "
         "Items must be unused, in original packaging, and accompanied by a receipt. "
         "Refunds are processed within 5-7 business days."},
        {"id": 2, "title": "Shipping Information", "content":
         "We offer Standard Shipping (5-7 business days, free on orders over $50), "
         "Expedited Shipping (2-3 business days, $12.99), and "
         "Overnight Shipping (next business day, $24.99)."},
    ]
    print(f"Using {len(DOCUMENTS)} fallback documents.")

-
## Step 2: Create Evaluation Dataset

We combine **manual golden test cases** with **synthetically generated** test cases for a comprehensive evaluation dataset.

In [ ]:
# Manual golden test cases (high quality, carefully crafted)
GOLDEN_TEST_CASES = [
    {
        "query": "What is the return policy for regular items?",
        "reference": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt. Refunds are processed in 5-7 business days.",
        "category": "returns",
    },
    {
        "query": "How long do I have to return electronics?",
        "reference": "Electronics have a 15-day return window. They must be returned with all original accessories and packaging. A 15% restocking fee may apply to opened items.",
        "category": "returns",
    },
    {
        "query": "What shipping options are available and how much do they cost?",
        "reference": "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping (2-3 days, $12.99), and Overnight Shipping (next business day, $24.99).",
        "category": "shipping",
    },
    {
        "query": "Do you ship internationally?",
        "reference": "Yes, Acme Corp ships to over 50 countries. Standard international shipping takes 10-21 business days. Customers are responsible for customs duties and import fees.",
        "category": "shipping",
    },
    {
        "query": "What does the warranty cover on Acme products?",
        "reference": "Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship. Does not cover accidents, misuse, or normal wear.",
        "category": "warranty",
    },
    {
        "query": "How can I contact customer support?",
        "reference": "Phone (1-800-ACME-HELP, M-F 8AM-8PM EST), email (support@acmecorp.com, 24-48hr response), live chat (M-Sat 9AM-6PM EST).",
        "category": "support",
    },
    {
        "query": "What are the features of the Acme SmartHome Hub?",
        "reference": "The SmartHome Hub costs $149.99, supports WiFi/Bluetooth/Zigbee/Z-Wave, has voice control, 5-inch touchscreen, energy monitoring, and automated routines. 2-year warranty.",
        "category": "products",
    },
    {
        "query": "How much is the AirPure Pro and what does it filter?",
        "reference": "The AirPure Pro costs $299.99. 4-stage filtration (pre-filter, carbon, True HEPA H13, UV-C). Removes 99.97% of particles 0.3 microns or larger. Covers rooms up to 800 sq ft.",
        "category": "products",
    },
    {
        "query": "What payment methods do you accept?",
        "reference": "Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. Orders over $200 qualify for Acme Pay Later.",
        "category": "payments",
    },
    {
        "query": "How does the Acme Rewards program work?",
        "reference": "Free to join, 1 point per dollar. 100 points = $5 off. Silver tier (500+ pts) adds free expedited shipping. Gold tier (1000+ pts) adds priority support.",
        "category": "loyalty",
    },
    {
        "query": "What is the electronics restocking fee?",
        "reference": "A restocking fee of 15% may apply to opened electronics returned to Acme Corp.",
        "category": "returns",
    },
    {
        "query": "How long does it take to process a refund?",
        "reference": "Refunds are processed within 5-7 business days after Acme Corp receives the returned item.",
        "category": "returns",
    },
]

print(f"Manual golden test cases: {len(GOLDEN_TEST_CASES)}")
categories = pd.Series([tc["category"] for tc in GOLDEN_TEST_CASES]).value_counts()
print(f"Categories: {dict(categories)}")

In [ ]:
# Synthetic test case generation using DeepEval Synthesizer
from deepeval.synthesizer import Synthesizer
from deepeval.test_case import LLMTestCase

# Prepare context for the synthesizer
context_list = [
    [f"{doc['title']}\n\n{doc['content']}"] for doc in DOCUMENTS
]

# Initialize synthesizer
synthesizer = Synthesizer(model="gpt-4o-mini")

# Generate synthetic test cases
synthetic_goldens = synthesizer.generate_goldens_from_contexts(
    contexts=context_list,
    max_goldens_per_context=1,
)

print(f"Generated {len(synthetic_goldens)} synthetic test cases")
for i, golden in enumerate(synthetic_goldens[:3]):
    print(f"\n  Sample {i+1}:")
    print(f"    Input: {golden.input[:80]}...")
    print(f"    Expected: {golden.expected_output[:80]}...")

In [ ]:
# Combine manual and synthetic test cases into a unified dataset

# Convert synthetic goldens to our format
synthetic_test_cases = []
for golden in synthetic_goldens:
    synthetic_test_cases.append({
        "query": golden.input,
        "reference": golden.expected_output,
        "category": "synthetic",
    })

# Combine
ALL_TEST_CASES = GOLDEN_TEST_CASES + synthetic_test_cases

print(f"Combined evaluation dataset: {len(ALL_TEST_CASES)} test cases")
print(f"  Manual: {len(GOLDEN_TEST_CASES)}")
print(f"  Synthetic: {len(synthetic_test_cases)}")

In [ ]:
# Build the default RAG pipeline
default_config = RAGConfig()
pipeline = RAGPipeline(config=default_config, documents=DOCUMENTS)

print(f"Default config: {default_config.name}")
print(f"Pipeline built with {pipeline.num_chunks} chunks from {len(DOCUMENTS)} documents")

In [ ]:
# Run all test cases through the pipeline and collect results
def run_evaluation_data(pipeline: RAGPipeline, test_cases: list[dict]) -> list[dict]:
    """Run test cases through a pipeline and collect evaluation data."""
    results = []
    for tc in test_cases:
        result = pipeline.run(tc["query"])
        results.append({
            "query": tc["query"],
            "response": result["answer"],
            "reference": tc["reference"],
            "contexts": result["contexts"],
            "category": tc.get("category", "unknown"),
            "latency_ms": result["latency_ms"],
        })
    return results

eval_data = run_evaluation_data(pipeline, ALL_TEST_CASES)
print(f"Collected {len(eval_data)} evaluation samples")
print(f"Average latency: {np.mean([d['latency_ms'] for d in eval_data]):.0f}ms")

-
## Step 3: Define Metrics Suite

We define a comprehensive metrics suite combining DeepEval, RAGAS, and custom metrics.

In [ ]:
# DeepEval metrics
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    GEval,
)
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# DeepEval RAG metrics
de_faithfulness = FaithfulnessMetric(model="gpt-4o-mini", threshold=0.7)
de_answer_relevancy = AnswerRelevancyMetric(model="gpt-4o-mini", threshold=0.7)
de_ctx_precision = ContextualPrecisionMetric(model="gpt-4o-mini", threshold=0.7)
de_ctx_recall = ContextualRecallMetric(model="gpt-4o-mini", threshold=0.7)
de_ctx_relevancy = ContextualRelevancyMetric(model="gpt-4o-mini", threshold=0.7)

# DeepEval G-Eval custom criteria: Domain-specific quality check
de_domain_quality = GEval(
    name="Domain Quality",
    model="gpt-4o-mini",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    evaluation_steps=[
        "Check if the response uses proper enterprise/business terminology.",
        "Verify the response includes specific numbers, names, or details when available.",
        "Ensure the response is clear and would be useful to an enterprise customer.",
        "Compare completeness against the expected output.",
    ],
    threshold=0.7,
)

de_metrics = {
    "de_faithfulness": de_faithfulness,
    "de_answer_relevancy": de_answer_relevancy,
    "de_ctx_precision": de_ctx_precision,
    "de_ctx_recall": de_ctx_recall,
    "de_ctx_relevancy": de_ctx_relevancy,
    "de_domain_quality": de_domain_quality,
}

print(f"DeepEval metrics defined: {len(de_metrics)}")
for name in de_metrics:
    print(f"  - {name}")

In [ ]:
# RAGAS metrics
from ragas import evaluate as ragas_evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    Faithfulness as RagasFaithfulness,
    ResponseRelevancy as RagasResponseRelevancy,
    LLMContextPrecisionWithReference as RagasContextPrecision,
    LLMContextRecall as RagasContextRecall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

ragas_metrics_list = [
    RagasFaithfulness(llm=ragas_llm),
    RagasResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    RagasContextPrecision(llm=ragas_llm),
    RagasContextRecall(llm=ragas_llm),
]

print(f"RAGAS metrics defined: {len(ragas_metrics_list)}")
for m in ragas_metrics_list:
    print(f"  - {m.name}")

-
## Step 4: Run Full Evaluation

Run both DeepEval and RAGAS evaluations on the full dataset.

In [ ]:
# Run DeepEval evaluation
print("Running DeepEval evaluation...")
print("=" * 60)

de_all_scores = {name: [] for name in de_metrics}

for idx, ed in enumerate(eval_data):
    tc = LLMTestCase(
        input=ed["query"],
        actual_output=ed["response"],
        expected_output=ed["reference"],
        retrieval_context=ed["contexts"],
    )
    
    for metric_name, metric in de_metrics.items():
        try:
            metric.measure(tc)
            de_all_scores[metric_name].append(metric.score)
        except Exception as e:
            print(f"  Warning: {metric_name} failed on sample {idx}: {e}")
            de_all_scores[metric_name].append(np.nan)
    
    if (idx + 1) % 5 == 0:
        print(f"  Processed {idx + 1}/{len(eval_data)} samples")

print(f"\nDeepEval evaluation complete. {len(eval_data)} samples evaluated.")

In [ ]:
# Run RAGAS evaluation
print("Running RAGAS evaluation...")
print("=" * 60)

ragas_samples = [
    SingleTurnSample(
        user_input=ed["query"],
        response=ed["response"],
        reference=ed["reference"],
        retrieved_contexts=ed["contexts"],
    )
    for ed in eval_data
]

ragas_dataset = EvaluationDataset(samples=ragas_samples)

ragas_results = ragas_evaluate(
    dataset=ragas_dataset,
    metrics=ragas_metrics_list,
)

ragas_df = ragas_results.to_pandas()
print(f"\nRAGAS evaluation complete.")
print(f"Aggregate: {ragas_results}")

-
## Step 5: Analyze Results

Combine all scores into a comprehensive DataFrame and perform analysis.

In [ ]:
# Build comprehensive results DataFrame
results_data = []

for idx, ed in enumerate(eval_data):
    row = {
        "query": ed["query"],
        "category": ed["category"],
        "latency_ms": ed["latency_ms"],
    }
    
    # Add DeepEval scores
    for metric_name in de_metrics:
        row[metric_name] = de_all_scores[metric_name][idx]
    
    # Add RAGAS scores
    ragas_row = ragas_df.iloc[idx]
    row["ragas_faithfulness"] = ragas_row.get("faithfulness", np.nan)
    row["ragas_relevancy"] = ragas_row.get("answer_relevancy", np.nan)
    row["ragas_ctx_precision"] = ragas_row.get("llm_context_precision_with_reference", np.nan)
    row["ragas_ctx_recall"] = ragas_row.get("llm_context_recall", np.nan)
    
    results_data.append(row)

results_df = pd.DataFrame(results_data)

# Display the full table
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Full Results DataFrame:")
print("=" * 100)
results_df

In [ ]:
# Summary statistics for all metrics
metric_cols = [
    "de_faithfulness", "de_answer_relevancy", "de_ctx_precision",
    "de_ctx_recall", "de_ctx_relevancy", "de_domain_quality",
    "ragas_faithfulness", "ragas_relevancy", "ragas_ctx_precision", "ragas_ctx_recall",
]

summary_stats = results_df[metric_cols].describe()
print("Summary Statistics:")
print("=" * 100)
print(summary_stats.round(3))

In [ ]:
# Identify worst-performing queries
print("Worst-Performing Queries (lowest average score):")
print("=" * 80)

results_df["avg_score"] = results_df[metric_cols].mean(axis=1)
worst_queries = results_df.nsmallest(5, "avg_score")[["query", "category", "avg_score"]]

for idx, row in worst_queries.iterrows():
    print(f"\n  Query: {row['query'][:60]}")
    print(f"  Category: {row['category']}")
    print(f"  Avg Score: {row['avg_score']:.3f}")
    # Show which metrics were lowest
    query_scores = results_df.loc[idx, metric_cols]
    worst_metric = query_scores.idxmin()
    print(f"  Worst Metric: {worst_metric} = {query_scores[worst_metric]:.3f}")

In [ ]:
# Correlation analysis between metrics
correlation = results_df[metric_cols].corr()

print("Metric Correlation Matrix:")
print("=" * 100)
print(correlation.round(3))

In [ ]:
# Cross-framework comparison: DeepEval vs RAGAS on equivalent metrics
framework_comparison = pd.DataFrame({
    "Metric": ["Faithfulness", "Answer Relevancy", "Ctx Precision", "Ctx Recall"],
    "DeepEval Mean": [
        results_df["de_faithfulness"].mean(),
        results_df["de_answer_relevancy"].mean(),
        results_df["de_ctx_precision"].mean(),
        results_df["de_ctx_recall"].mean(),
    ],
    "RAGAS Mean": [
        results_df["ragas_faithfulness"].mean(),
        results_df["ragas_relevancy"].mean(),
        results_df["ragas_ctx_precision"].mean(),
        results_df["ragas_ctx_recall"].mean(),
    ],
})
framework_comparison["Difference"] = (
    framework_comparison["DeepEval Mean"] - framework_comparison["RAGAS Mean"]
)

print("Framework Comparison (DeepEval vs RAGAS):")
print("=" * 70)
print(framework_comparison.round(3).to_string(index=False))

In [ ]:
# Performance by category
category_analysis = results_df.groupby("category")[metric_cols + ["latency_ms"]].mean()

print("Performance by Category:")
print("=" * 100)
print(category_analysis.round(3))

In [ ]:
# Visualization: Bar chart of average scores by metric
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: Average scores by metric
avg_scores = results_df[metric_cols].mean().sort_values(ascending=True)
colors = ["#4CAF50" if s >= 0.7 else "#FF9800" if s >= 0.5 else "#F44336" for s in avg_scores]
axes[0, 0].barh(range(len(avg_scores)), avg_scores.values, color=colors)
axes[0, 0].set_yticks(range(len(avg_scores)))
axes[0, 0].set_yticklabels(avg_scores.index, fontsize=8)
axes[0, 0].set_xlabel("Average Score")
axes[0, 0].set_title("Average Score by Metric")
axes[0, 0].axvline(x=0.7, color="gray", linestyle="--", alpha=0.5, label="Threshold")
axes[0, 0].legend()

# Chart 2: Score distribution (box plot)
results_df[metric_cols[:6]].boxplot(ax=axes[0, 1], rot=45)
axes[0, 1].set_title("DeepEval Score Distributions")
axes[0, 1].set_ylabel("Score")

# Chart 3: RAGAS score distribution
ragas_cols = [c for c in metric_cols if c.startswith("ragas_")]
results_df[ragas_cols].boxplot(ax=axes[1, 0], rot=45)
axes[1, 0].set_title("RAGAS Score Distributions")
axes[1, 0].set_ylabel("Score")

# Chart 4: Latency distribution
axes[1, 1].hist(results_df["latency_ms"], bins=15, edgecolor="black", alpha=0.7)
axes[1, 1].set_xlabel("Latency (ms)")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("Query Latency Distribution")
axes[1, 1].axvline(x=results_df["latency_ms"].mean(), color="red", linestyle="--",
                    label=f"Mean: {results_df['latency_ms'].mean():.0f}ms")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("evaluation_results.png", dpi=100, bbox_inches="tight")
plt.show()
print("Charts saved to evaluation_results.png")

In [ ]:
# Heatmap of metric correlations
fig, ax = plt.subplots(figsize=(12, 10))

corr = results_df[metric_cols].corr()
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)

ax.set_xticks(range(len(metric_cols)))
ax.set_yticks(range(len(metric_cols)))
ax.set_xticklabels(metric_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(metric_cols, fontsize=8)

# Add correlation values
for i in range(len(metric_cols)):
    for j in range(len(metric_cols)):
        val = corr.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im)
ax.set_title("Metric Correlation Heatmap")
plt.tight_layout()
plt.savefig("metric_correlations.png", dpi=100, bbox_inches="tight")
plt.show()
print("Heatmap saved to metric_correlations.png")

-
## Step 6: Hyperparameter Experiments

Test different RAG configurations to find the optimal settings.

In [ ]:
# Define experiment configurations
# We use a subset of test cases for faster experimentation
EXPERIMENT_TEST_CASES = GOLDEN_TEST_CASES[:6]  # Use first 6 for speed

# Key metrics to track (using DeepEval for speed)
def evaluate_config(config: RAGConfig) -> dict:
    """Evaluate a single RAG configuration."""
    pipe = RAGPipeline(config=config, documents=DOCUMENTS)
    eval_data = run_evaluation_data(pipe, EXPERIMENT_TEST_CASES)
    
    faith_scores = []
    relev_scores = []
    recall_scores = []
    latencies = []
    
    for ed in eval_data:
        tc = LLMTestCase(
            input=ed["query"],
            actual_output=ed["response"],
            expected_output=ed["reference"],
            retrieval_context=ed["contexts"],
        )
        
        f_metric = FaithfulnessMetric(model="gpt-4o-mini", threshold=0.7)
        r_metric = AnswerRelevancyMetric(model="gpt-4o-mini", threshold=0.7)
        c_metric = ContextualRecallMetric(model="gpt-4o-mini", threshold=0.7)
        
        f_metric.measure(tc)
        r_metric.measure(tc)
        c_metric.measure(tc)
        
        faith_scores.append(f_metric.score)
        relev_scores.append(r_metric.score)
        recall_scores.append(c_metric.score)
        latencies.append(ed["latency_ms"])
    
    return {
        "config": config.name,
        "chunk_size": config.chunk_size,
        "top_k": config.top_k,
        "temperature": config.temperature,
        "num_chunks": pipe.num_chunks,
        "faithfulness": np.mean(faith_scores),
        "relevancy": np.mean(relev_scores),
        "ctx_recall": np.mean(recall_scores),
        "avg_latency_ms": np.mean(latencies),
    }

print(f"Experiment framework defined. Using {len(EXPERIMENT_TEST_CASES)} test cases per config.")

In [ ]:
# Experiment 1: Chunk Size
print("Experiment 1: Chunk Size")
print("=" * 60)

chunk_size_configs = [
    RAGConfig(chunk_size=256, top_k=3, temperature=0.0),
    RAGConfig(chunk_size=512, top_k=3, temperature=0.0),
    RAGConfig(chunk_size=1024, top_k=3, temperature=0.0),
]

chunk_results = []
for config in chunk_size_configs:
    print(f"\n  Testing chunk_size={config.chunk_size}...")
    result = evaluate_config(config)
    chunk_results.append(result)
    print(f"    Faithfulness: {result['faithfulness']:.3f}")
    print(f"    Relevancy:    {result['relevancy']:.3f}")
    print(f"    Ctx Recall:   {result['ctx_recall']:.3f}")
    print(f"    Chunks: {result['num_chunks']}, Latency: {result['avg_latency_ms']:.0f}ms")

chunk_df = pd.DataFrame(chunk_results)
print("\nChunk Size Experiment Results:")
print(chunk_df[["chunk_size", "num_chunks", "faithfulness", "relevancy", "ctx_recall", "avg_latency_ms"]].round(3))

In [ ]:
# Experiment 2: Top-K
print("Experiment 2: Top-K")
print("=" * 60)

topk_configs = [
    RAGConfig(chunk_size=500, top_k=3, temperature=0.0),
    RAGConfig(chunk_size=500, top_k=5, temperature=0.0),
    RAGConfig(chunk_size=500, top_k=10, temperature=0.0),
]

topk_results = []
for config in topk_configs:
    print(f"\n  Testing top_k={config.top_k}...")
    result = evaluate_config(config)
    topk_results.append(result)
    print(f"    Faithfulness: {result['faithfulness']:.3f}")
    print(f"    Relevancy:    {result['relevancy']:.3f}")
    print(f"    Ctx Recall:   {result['ctx_recall']:.3f}")

topk_df = pd.DataFrame(topk_results)
print("\nTop-K Experiment Results:")
print(topk_df[["top_k", "faithfulness", "relevancy", "ctx_recall", "avg_latency_ms"]].round(3))

In [ ]:
# Experiment 3: Temperature
print("Experiment 3: Temperature")
print("=" * 60)

temp_configs = [
    RAGConfig(chunk_size=500, top_k=3, temperature=0.0),
    RAGConfig(chunk_size=500, top_k=3, temperature=0.3),
    RAGConfig(chunk_size=500, top_k=3, temperature=0.7),
]

temp_results = []
for config in temp_configs:
    print(f"\n  Testing temperature={config.temperature}...")
    result = evaluate_config(config)
    temp_results.append(result)
    print(f"    Faithfulness: {result['faithfulness']:.3f}")
    print(f"    Relevancy:    {result['relevancy']:.3f}")
    print(f"    Ctx Recall:   {result['ctx_recall']:.3f}")

temp_df = pd.DataFrame(temp_results)
print("\nTemperature Experiment Results:")
print(temp_df[["temperature", "faithfulness", "relevancy", "ctx_recall", "avg_latency_ms"]].round(3))

In [ ]:
# Find optimal configuration
all_experiment_results = chunk_results + topk_results + temp_results
all_exp_df = pd.DataFrame(all_experiment_results)

# Score each configuration by average of all metrics
all_exp_df["avg_quality"] = all_exp_df[["faithfulness", "relevancy", "ctx_recall"]].mean(axis=1)

# Sort by average quality
all_exp_df_sorted = all_exp_df.sort_values("avg_quality", ascending=False)

print("All Configurations Ranked by Average Quality:")
print("=" * 100)
print(all_exp_df_sorted[[
    "config", "chunk_size", "top_k", "temperature",
    "faithfulness", "relevancy", "ctx_recall", "avg_quality", "avg_latency_ms"
]].round(3).to_string(index=False))

best_config = all_exp_df_sorted.iloc[0]
print(f"\nOptimal Configuration:")
print(f"  Chunk Size:  {int(best_config['chunk_size'])}")
print(f"  Top-K:       {int(best_config['top_k'])}")
print(f"  Temperature: {best_config['temperature']:.1f}")
print(f"  Avg Quality: {best_config['avg_quality']:.3f}")
print(f"  Avg Latency: {best_config['avg_latency_ms']:.0f}ms")

-
## Step 7: Regression Testing

Save baseline scores and compare future runs against them to detect regressions.

In [ ]:
# Define baseline from current results
baseline = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "chunk_size": default_config.chunk_size,
        "top_k": default_config.top_k,
        "temperature": default_config.temperature,
        "generation_model": default_config.generation_model,
        "embedding_model": default_config.embedding_model,
    },
    "scores": {
        "de_faithfulness": float(results_df["de_faithfulness"].mean()),
        "de_answer_relevancy": float(results_df["de_answer_relevancy"].mean()),
        "de_ctx_precision": float(results_df["de_ctx_precision"].mean()),
        "de_ctx_recall": float(results_df["de_ctx_recall"].mean()),
        "de_ctx_relevancy": float(results_df["de_ctx_relevancy"].mean()),
        "de_domain_quality": float(results_df["de_domain_quality"].mean()),
        "ragas_faithfulness": float(results_df["ragas_faithfulness"].mean()),
        "ragas_relevancy": float(results_df["ragas_relevancy"].mean()),
        "ragas_ctx_precision": float(results_df["ragas_ctx_precision"].mean()),
        "ragas_ctx_recall": float(results_df["ragas_ctx_recall"].mean()),
    },
    "num_test_cases": len(ALL_TEST_CASES),
}

# Save baseline to JSON
baseline_path = os.path.join("..", "baseline_scores.json")
with open(baseline_path, "w") as f:
    json.dump(baseline, f, indent=2)

print(f"Baseline saved to {baseline_path}")
print(f"\nBaseline scores:")
for metric, score in baseline["scores"].items():
    print(f"  {metric}: {score:.3f}")

In [ ]:
# Regression testing function
def check_regression(current_scores: dict, baseline_path: str, threshold: float = 0.05) -> dict:
    """Compare current scores against baseline and flag regressions.
    
    A regression is flagged when a metric drops by more than `threshold` 
    compared to the baseline.
    
    Args:
        current_scores: Dict of {metric_name: current_mean_score}
        baseline_path: Path to baseline JSON file
        threshold: Maximum allowed decrease before flagging (default 0.05 = 5%)
    
    Returns:
        Dict with regression analysis results
    """
    with open(baseline_path, "r") as f:
        baseline_data = json.load(f)
    
    baseline_scores = baseline_data["scores"]
    
    results = {
        "regressions": [],
        "improvements": [],
        "stable": [],
        "overall_passed": True,
    }
    
    for metric, current in current_scores.items():
        if metric not in baseline_scores:
            continue
        
        baseline_val = baseline_scores[metric]
        diff = current - baseline_val
        
        if diff < -threshold:
            results["regressions"].append({
                "metric": metric,
                "baseline": baseline_val,
                "current": current,
                "change": diff,
            })
            results["overall_passed"] = False
        elif diff > threshold:
            results["improvements"].append({
                "metric": metric,
                "baseline": baseline_val,
                "current": current,
                "change": diff,
            })
        else:
            results["stable"].append({
                "metric": metric,
                "baseline": baseline_val,
                "current": current,
                "change": diff,
            })
    
    return results

print("Regression testing function defined.")

In [ ]:
# Demo regression test (comparing against itself -- should show all stable)
current_scores = {
    metric: float(results_df[metric].mean())
    for metric in metric_cols
}

regression_results = check_regression(current_scores, baseline_path, threshold=0.05)

print("Regression Test Results:")
print("=" * 60)
print(f"Overall: {'PASSED' if regression_results['overall_passed'] else 'FAILED'}")
print(f"\nRegressions:  {len(regression_results['regressions'])}")
for r in regression_results["regressions"]:
    print(f"  FAIL: {r['metric']} -- baseline={r['baseline']:.3f}, current={r['current']:.3f}, change={r['change']:+.3f}")

print(f"\nImprovements: {len(regression_results['improvements'])}")
for r in regression_results["improvements"]:
    print(f"  BETTER: {r['metric']} -- baseline={r['baseline']:.3f}, current={r['current']:.3f}, change={r['change']:+.3f}")

print(f"\nStable:       {len(regression_results['stable'])}")
for r in regression_results["stable"]:
    print(f"  OK: {r['metric']} -- baseline={r['baseline']:.3f}, current={r['current']:.3f}, change={r['change']:+.3f}")

In [ ]:
# Simulate a regression scenario to show what detection looks like
simulated_scores = current_scores.copy()
simulated_scores["de_faithfulness"] = current_scores["de_faithfulness"] - 0.12  # Simulated drop
simulated_scores["ragas_ctx_recall"] = current_scores["ragas_ctx_recall"] - 0.08  # Simulated drop

sim_regression = check_regression(simulated_scores, baseline_path, threshold=0.05)

print("Simulated Regression Scenario:")
print("=" * 60)
print(f"Overall: {'PASSED' if sim_regression['overall_passed'] else 'FAILED'}")

if sim_regression["regressions"]:
    print("\nDetected regressions:")
    for r in sim_regression["regressions"]:
        print(f"  REGRESSION: {r['metric']}")
        print(f"    Baseline: {r['baseline']:.3f}")
        print(f"    Current:  {r['current']:.3f}")
        print(f"    Change:   {r['change']:+.3f}")

-
## Step 8: Generate Report

Create a comprehensive evaluation report as a markdown summary.

In [ ]:
def generate_evaluation_report(
    results_df: pd.DataFrame,
    metric_cols: list[str],
    experiment_results: pd.DataFrame,
    baseline: dict,
) -> str:
    """Generate a comprehensive evaluation report."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Calculate key stats
    avg_scores = results_df[metric_cols].mean()
    failing_metrics = [m for m in metric_cols if avg_scores[m] < 0.7]
    
    report = f"""# RAG Evaluation Report

**Generated**: {timestamp}
**Pipeline Config**: chunk_size={baseline['config']['chunk_size']}, top_k={baseline['config']['top_k']}, temp={baseline['config']['temperature']}
**Models**: Embedding={baseline['config']['embedding_model']}, Generation={baseline['config']['generation_model']}
**Test Cases**: {len(results_df)} ({len([tc for tc in ALL_TEST_CASES if tc['category'] != 'synthetic'])} manual + {len([tc for tc in ALL_TEST_CASES if tc['category'] == 'synthetic'])} synthetic)

---

## Executive Summary

The RAG pipeline was evaluated across {len(metric_cols)} metrics from both DeepEval and RAGAS frameworks.

**Overall Quality**: {'GOOD' if len(failing_metrics) == 0 else 'NEEDS IMPROVEMENT'}
- Metrics meeting threshold (>= 0.7): {len(metric_cols) - len(failing_metrics)}/{len(metric_cols)}
- Metrics below threshold: {len(failing_metrics)}
{'- Failing: ' + ', '.join(failing_metrics) if failing_metrics else ''}

## Metric Scores

| Metric | Mean | Min | Max | Std | Status |
|--------|------|-----|-----|-----|--------|
"""
    
    for metric in metric_cols:
        mean_val = results_df[metric].mean()
        min_val = results_df[metric].min()
        max_val = results_df[metric].max()
        std_val = results_df[metric].std()
        status = "PASS" if mean_val >= 0.7 else "FAIL"
        report += f"| {metric} | {mean_val:.3f} | {min_val:.3f} | {max_val:.3f} | {std_val:.3f} | {status} |\n"
    
    # Add worst queries section
    results_df_copy = results_df.copy()
    results_df_copy["avg_score"] = results_df_copy[metric_cols].mean(axis=1)
    worst = results_df_copy.nsmallest(3, "avg_score")
    
    report += f"""
## Worst-Performing Queries

| Query | Category | Avg Score |
|-------|----------|----------|
"""
    for _, row in worst.iterrows():
        report += f"| {row['query'][:50]} | {row['category']} | {row['avg_score']:.3f} |\n"
    
    # Add experiment findings
    best = experiment_results.iloc[0]
    report += f"""
## Hyperparameter Experiment Findings

**Optimal configuration found**: chunk_size={int(best['chunk_size'])}, top_k={int(best['top_k'])}, temperature={best['temperature']:.1f}
- Average quality: {best['avg_quality']:.3f}
- Average latency: {best['avg_latency_ms']:.0f}ms

## Recommendations

1. **Retriever**: {'Context recall is strong (>0.8). No changes needed.' if avg_scores.get('de_ctx_recall', 0) >= 0.8 else 'Context recall needs improvement. Consider increasing top-K or improving embeddings.'}
2. **Generator**: {'Faithfulness is high (>0.85). LLM grounding is working well.' if avg_scores.get('de_faithfulness', 0) >= 0.85 else 'Faithfulness needs improvement. Strengthen grounding instructions in the system prompt.'}
3. **Temperature**: {'Recommend temperature=0.0 for factual accuracy.' if best['temperature'] == 0.0 else f'Current optimal temperature is {best["temperature"]}.'}
4. **Monitoring**: Set up automated regression testing with threshold of 0.05 for all metrics.

---
*Report generated by RAG Evaluation Pipeline v1.0*
"""
    return report

print("Report generator defined.")

In [ ]:
# Generate the report
report = generate_evaluation_report(
    results_df=results_df,
    metric_cols=metric_cols,
    experiment_results=all_exp_df_sorted,
    baseline=baseline,
)

# Save the report
report_path = os.path.join("..", "evaluation_report.md")
with open(report_path, "w") as f:
    f.write(report)

print(f"Report saved to {report_path}")
print("\n" + "=" * 80)
print(report)

In [ ]:
# Save all results to CSV for further analysis
results_csv_path = os.path.join("..", "evaluation_results.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"Full results saved to {results_csv_path}")

experiments_csv_path = os.path.join("..", "experiment_results.csv")
all_exp_df_sorted.to_csv(experiments_csv_path, index=False)
print(f"Experiment results saved to {experiments_csv_path}")

-
## Complete Pipeline Summary

This notebook demonstrated a **complete, production-ready RAG evaluation pipeline**:

### What We Built

1. **Configurable RAG Pipeline** -- `RAGPipeline` class with adjustable chunk_size, top_k, temperature, and model parameters

2. **Comprehensive Evaluation Dataset** -- 12+ manual golden test cases combined with synthetic test cases from DeepEval Synthesizer

3. **Full Metrics Suite**:
   - DeepEval: Faithfulness, Answer Relevancy, Contextual Precision/Recall/Relevancy, Domain Quality (G-Eval)
   - RAGAS: Faithfulness, Response Relevancy, Context Precision, Context Recall

4. **Dual-Framework Evaluation** -- Both DeepEval and RAGAS running on the same dataset

5. **Comprehensive Analysis** -- Summary statistics, worst-performer identification, correlation analysis, category breakdown, visualizations

6. **Hyperparameter Optimization** -- Systematic testing of chunk_size, top_k, and temperature

7. **Regression Testing** -- Baseline saving and comparison with configurable thresholds

8. **Automated Reporting** -- Markdown report generation with findings and recommendations

### Production Deployment Checklist

- [ ] Evaluation dataset covers all query types and edge cases
- [ ] All metrics exceed minimum thresholds
- [ ] Optimal hyperparameters identified and documented
- [ ] Baseline scores saved for regression testing
- [ ] Regression testing integrated into CI/CD pipeline
- [ ] Monitoring dashboards set up for real-time metrics
- [ ] Alert system configured for metric regressions
- [ ] Regular evaluation cadence established (weekly/monthly)

### Files Generated

- `baseline_scores.json` -- Baseline metric scores for regression testing
- `evaluation_report.md` -- Summary report with findings
- `evaluation_results.csv` -- Full per-query metric scores
- `experiment_results.csv` -- Hyperparameter experiment results
- `evaluation_results.png` -- Score distribution charts
- `metric_correlations.png` -- Metric correlation heatmap

This completes the RAG Evaluation notebook series. You now have the knowledge and tools to build comprehensive evaluation pipelines for any RAG system.